# ConfigSelection Basics

This tutorial demonstrates basic capabilities of the ConfigSelection class.

A ConfigSelection allows iterating over configurations stored in an enumeration's [configuration_set](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.configuration_set.html#casm.project.enum.EnumData.configuration_set) and [configuration_list](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.configuration_list.html#casm.project.enum.EnumData.configuration_list) in a single pass. It handles accessing project data to make it easier to calculate configuration properties such as the parametric composition or correlations for a particular basis set. 

The ConfigSelection can also be helpful when setting up calculations, handling job submission, and reporting calculation results.

A ConfigSelection stores whether configurations are "selected" or "unselected" in a file in the enumeration directory so it can be easily saved, loaded, and updated.


# Setup

The first few cells construct a Si-Ge project and perform an enumeration.

In [1]:
import pathlib

import libcasm.xtal as xtal
from casm.project import Project
from casm.tools.shared.json_io import safe_dump

input_dir = pathlib.Path("input")

project_path = pathlib.Path("ConfigSelection_basics/SiGe_occ")
project_path.mkdir(parents=True, exist_ok=True)

## Project initialization


In [2]:
prim_data = {
    "title": "SiGe_occ",
    "lattice_vectors": [
        [0.000000000000, 2.800000000000, 2.800000000000],  # 1st lattice vector
        [2.800000000000, 0.000000000000, 2.800000000000],  # 2nd lattice vector
        [2.800000000000, 2.800000000000, 0.000000000000],  # 3rd lattice vector
    ],
    "coordinate_mode": "Fractional",
    "basis": [
        {
            "coordinate": [0.0, 0.0, 0.0],
            "occupant_dof": ["Si", "Ge"],
        },
        {
            "coordinate": [0.25, 0.25, 0.25],
            "occupant_dof": ["Si", "Ge"],
        },
    ],
}

with open(project_path / "prim.json", "w") as f:
    f.write(xtal.pretty_json(prim_data))

In [3]:
project = Project.init(path=project_path)

CASM project already exists at ConfigSelection_basics/SiGe_occ
Using existing project


## Enumerate configurations
    

In [4]:
# Enumerate configurations in supercells with volume 1 to 4
enum = project.enum.get("occ_by_supercell.1")
enum.occ_by_supercell(max=4, min=1)

-- Begin: Enumerating occupations by supercell --

Enumerate configurations for: SCEL1_1_1_1_0_0_0
3 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL2_2_1_1_0_1_1
4 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL2_2_1_1_0_0_1
3 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL3_3_1_1_0_2_2
13 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL3_3_1_1_0_2_1
10 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL3_3_1_1_0_0_2
10 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL4_4_1_1_0_0_0
36 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL4_4_1_1_0_1_0
27 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL4_4_1_1_0_0_2
36 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL4_4_1_1_0_0_3
24 configurations (0 new, 0 exc

In [5]:
project.enum.merge(
    src_id="supercells_by_volume.1",
    dest_id="supercells_by_volume.3",
)
project.enum.list()

EnumData:
- id: occ_by_supercell.1
- supercell_set: 13 supercells
- configuration_set: 214 configurations
- configuration_list: 10 configurations
EnumData:
- id: supercells_by_volume.3


## Calculation settings for ASE + VASP

This adds calculation settings which will be used in demonstrating how a ConfigSelection helps setting up and managing calculations.

In [6]:
import os

# Give your calculation settings a name
calctype_id = "vasp-parameter-set-1"

# Create a new clex (CLuster EXpansion) description for
# a cluster expansion that uses the calctype, and set it as the default
project.settings.add_clex(name=calctype_id, calctype=calctype_id)
project.settings.set_default_clex(name=calctype_id)

# A helper for setting up and collecting calculation files:
calc = project.calc.get(calctype_id)

### Write template INCAR file
#
# !! CHANGE THIS AS NECESSARY !!
# Used in calculation setup as input to ase.calculators.vasp.Vasp.read_incar
#
incar_text = """\
ISPIN = 1 #does non spin-polarized calc.
PREC = Accurate #cutoff + wrap around errors.
IBRION= 2 #conj. grad. relaxation.
NSW=61 #numberof ionic steps taken in minimization. Make it odd.
ISIF= 3 #whether stress tensor is calculated, what is allowed to relax.
ENMAX=600 #cutoff
ISMEAR = 1 #BZ integration method (for relaxation runs).
SIGMA = 0.2 #smearing width (keep T*S < 1meV/atom). 
LWAVE = .FALSE.
LCHARG = .FALSE.
"""
calc.write_text_file(
    name="INCAR",
    text=incar_text,
)

### Write template KPOINTS file
#
# !! CHANGE THIS AS NECESSARY !!
# Used in calculation setup as input to ase.calculators.vasp.Vasp.read_kpoints
#
kpoints_text = """\
Fully automatic mesh
0              ! 0 -> automatic generation scheme 
Auto           ! fully automatic
  71           ! length (R_k)
"""
calc.write_text_file(
    name="KPOINTS",
    text=kpoints_text,
)

### Write settings for ase.calculators.vasp.Vasp in "calc.json"
# !! CHANGE THIS AS NECESSARY !!
calc_settings = {
    "setups": {
        "Si": "",
        "Ge": "_d",
    },
    "xc": "pbe",
}
calc.write_json_file(
    name="calc.json",
    data=calc_settings,
)

# !! CHANGE THIS AS NECESSARY !!
# write ASE config.ini file, set
# VASP_PP_PATH for ase.calculators.vasp.Vasp
vasp_pp_path = input_dir / "dummy_vasp_potentials/"
config_ini_path = input_dir / "config.ini"
config_ini_path.write_text(
    f"[environment]\nVASP_PP_PATH = {str(vasp_pp_path.resolve())}\n"
)
os.environ["ASE_CONFIG_PATH"] = str(config_ini_path.resolve())


overwrite: ConfigSelection_basics/SiGe_occ/calculation_settings/calctype.vasp-parameter-set-1/INCAR
overwrite: ConfigSelection_basics/SiGe_occ/calculation_settings/calctype.vasp-parameter-set-1/KPOINTS
overwrite: ConfigSelection_basics/SiGe_occ/calculation_settings/calctype.vasp-parameter-set-1/calc.json


### Calculation input files

The calculation directories created will be at:

    <enum_dir>/training_data/calctype.<calctype_id>/<configname>/

Where

    <enum_dir> = <project>/enumerations/enum.<enum_id>/



In [7]:
# Add some configurations to the configuration_list
enum_id = "occ_by_supercell.1"
enum = project.enum.get(enum_id)
config_selection = enum.config_selection(name="main")
print("len(config_selection):", len(config_selection))
for record in config_selection:
    if record.configuration.supercell.n_unitcells < 3:
        config = record.configuration
        if config not in enum.configuration_list:
            enum.configuration_list.append(config)
enum.commit()
config_selection = enum.config_selection(name="main")
print("len(config_selection):", len(config_selection))
print()

print("Insert all...")
config_selection.insert_all(selected=False)
print("len(config_selection):", len(config_selection))
print("n_selected:", config_selection.n_selected)
print("n_unselected:", config_selection.n_unselected)
print()

# print("Clear and clean...")
# enum.configuration_list.clear()
# config_selection.clean()
# print("len(config_selection):", len(config_selection))
# print("n_selected:", config_selection.n_selected)
# print("n_unselected:", config_selection.n_unselected)
# print()

len(config_selection): 224
overwrite: ConfigSelection_basics/SiGe_occ/enumerations/enum.occ_by_supercell.1/scel_set.json
overwrite: ConfigSelection_basics/SiGe_occ/enumerations/enum.occ_by_supercell.1/config_set.json
overwrite: ConfigSelection_basics/SiGe_occ/enumerations/enum.occ_by_supercell.1/config_list.json
len(config_selection): 224

Insert all...
len(config_selection): 224
n_selected: 224
n_unselected: 0



In [8]:
# Enumeration ID & Calculation type ID to use
enum_id = "occ_by_supercell.1"
calctype_id = "vasp-parameter-set-1"

# Create a ConfigSelection for iterating over configurations in an enumeration
# By default, all configurations are selected
enum = project.enum.get(enum_id)

config_selection = enum.config_selection(name="main")
print("len(config_selection):", len(config_selection))

# Setup calculations using the chosen calculation type
calc = project.calc.get(calctype_id)
calc.setup(
    config_selection=config_selection,
    tool="vasp",
)

len(config_selection): 224
skipping: SCEL1_1_1_1_0_0_0/0 (status=setup)
skipping: SCEL1_1_1_1_0_0_0/1 (status=setup)
skipping: SCEL1_1_1_1_0_0_0/2 (status=setup)
skipping: SCEL2_2_1_1_0_1_1/0 (status=setup)
skipping: SCEL2_2_1_1_0_1_1/3 (status=setup)
skipping: SCEL2_2_1_1_0_1_1/1 (status=setup)
skipping: SCEL2_2_1_1_0_1_1/2 (status=setup)
skipping: SCEL2_2_1_1_0_0_1/0 (status=setup)
skipping: SCEL2_2_1_1_0_0_1/1 (status=setup)
skipping: SCEL2_2_1_1_0_0_1/2 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/0 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/9 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/5 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/2 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/1 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/10 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/6 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/12 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/3 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/11 (status=setup)
skipping: SCEL3_3_1_1_0_2_2/7 (status=setup)
skipping: SCEL3_3_1_1_0_2

In [9]:
# Check calculation directories

from casm.tools.shared.json_io import printpathstr

for record in config_selection:
    # print("~~~")
    # print("calc_dir:", printpathstr(record.calc_dir))
    # print("files:", os.listdir(record.calc_dir))
    print(f"{record.name}: {record.supercell_name} {record.n_unitcells}")

# for record in config_selection:
#     record.set_selected(record.name == "config_list/6")
# config_selection.deselect_if(lambda record: record.name == "config_list/7")
# print(config_selection.n_selected)

SCEL1_1_1_1_0_0_0/0: SCEL1_1_1_1_0_0_0 1
SCEL1_1_1_1_0_0_0/1: SCEL1_1_1_1_0_0_0 1
SCEL1_1_1_1_0_0_0/2: SCEL1_1_1_1_0_0_0 1
SCEL2_2_1_1_0_1_1/0: SCEL2_2_1_1_0_1_1 2
SCEL2_2_1_1_0_1_1/3: SCEL2_2_1_1_0_1_1 2
SCEL2_2_1_1_0_1_1/1: SCEL2_2_1_1_0_1_1 2
SCEL2_2_1_1_0_1_1/2: SCEL2_2_1_1_0_1_1 2
SCEL2_2_1_1_0_0_1/0: SCEL2_2_1_1_0_0_1 2
SCEL2_2_1_1_0_0_1/1: SCEL2_2_1_1_0_0_1 2
SCEL2_2_1_1_0_0_1/2: SCEL2_2_1_1_0_0_1 2
SCEL3_3_1_1_0_2_2/0: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/9: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/5: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/2: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/1: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/10: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/6: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/12: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/3: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/11: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/7: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/4: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_2_2/8: SCEL3_3_1_1_0_2_2 3
SCEL3_3_1_1_0_0_2/0: SCEL3_3_1_1_0_0_2 3
SCEL3_3_1_1_0

In [10]:
# import subprocess
# p = subprocess.run("echo ${HOME}", shell=True)

os.environ["CASM_PROJ_PATH"] = str(project.path) + "/"
config_selection.get("config_list/8").run_subprocess(
    args=[
        'echo "${PWD/#$CASM_PROJ_PATH/}:"',
        "ls -hl",
    ],
    # args=["pwd"],
    write_log=False,
    shell=True,
)

enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1/config_list/8:
total 72
-rw-r--r--  1 bpuchala  staff    52B Jul 24  2025 ase-sort.dat
-rw-r--r--  1 bpuchala  staff   201B Jul 24  2025 config.json
-rw-r--r--  1 bpuchala  staff   144B Jul 24  2025 INCAR
-rw-r--r--  1 bpuchala  staff    65B Jul 24  2025 KPOINTS
-rw-r--r--  1 bpuchala  staff   502B Jul 24  2025 POSCAR
-rw-r--r--  1 bpuchala  staff    18B Jul 24  2025 POTCAR
-rw-r--r--  1 bpuchala  staff    24B Jul 24  2025 status.json
-rw-r--r--  1 bpuchala  staff   406B Jul 24  2025 structure.json
-rw-r--r--  1 bpuchala  staff    69B Feb 28 11:31 test.sh


[CompletedProcess(args='echo "${PWD/#$CASM_PROJ_PATH/}:"', returncode=0),
 CompletedProcess(args='ls -hl', returncode=0)]

In [11]:
from casm.tools.calc.scripts import get_script_path

config_selection.run_shell_script(
    script=get_script_path("vasp/test.sh"),
    write_log=False,
)

~~~ vasp/test.sh ~~~
cwd: /Users/bpuchala/codes/CASM_v2_source/CASMcode_modules/CASMcode_project/notebooks/ConfigSelection_basics/SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1/SCEL1_1_1_1_0_0_0/0
ls -hl:
total 72
-rw-r--r--  1 bpuchala  staff    26B Jul 24  2025 ase-sort.dat
-rw-r--r--  1 bpuchala  staff   192B Jul 24  2025 config.json
-rw-r--r--  1 bpuchala  staff   144B Jul 24  2025 INCAR
-rw-r--r--  1 bpuchala  staff    65B Jul 24  2025 KPOINTS
-rw-r--r--  1 bpuchala  staff   369B Jul 24  2025 POSCAR
-rw-r--r--  1 bpuchala  staff     8B Jul 24  2025 POTCAR
-rw-r--r--  1 bpuchala  staff    24B Jul 24  2025 status.json
-rw-r--r--  1 bpuchala  staff   255B Jul 24  2025 structure.json
-rw-r--r--  1 bpuchala  staff    69B Feb 28 11:32 test.sh
~~~ vasp/test.sh ~~~
cwd: /Users/bpuchala/codes/CASM_v2_source/CASMcode_modules/CASMcode_project/notebooks/ConfigSelection_basics/SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-p